# Redes Neuronales para Predecir Porosidad
## Módulo 6 · Día 1 · Construcción de un MLP con Keras · UDLA

**Instructor: David Ponce**

---

### ¿Qué aprenderemos hoy?

Hasta ahora usamos modelos de ML clásico (K-Means, DBSCAN, PCA). Hoy damos
el salto a las **redes neuronales**: máquinas que aprenden patrones complejos
por sí mismas, sin que les demos las reglas.

Nuestra misión: **predecir la porosidad (NPHI) de la roca** a partir de otras
3 curvas geofísicas (GR, ILD, RHOB). Es un problema de **regresión** — predecir
un valor continuo.

### Objetivos de la sesión:
1. Entender qué es una neurona artificial (pesos + sesgo + activación)
2. Construir un Perceptrón Multicapa (MLP) con Keras
3. Entrenar la red para predecir porosidad y evaluar su error (MSE)
4. Visualizar la porosidad predicha vs la real

> **Cómo usar este notebook:** Cada celda empieza con una explicación en
> lenguaje simple de QUÉ hace, y el código está comentado línea por línea
> con el PORQUÉ. Ejecuta una celda a la vez y lee antes de correr.

---
## PARTE 1: Importando las herramientas

Hoy estrenamos **TensorFlow/Keras**, la librería de deep learning de Google.
Keras es una interfaz de alto nivel que hace que construir redes neuronales
sea tan simple como apilar bloques de Lego.

En lenguaje simple: así como antes importábamos pandas para tablas y sklearn
para K-Means, hoy importamos TensorFlow para redes neuronales.

### Celda 1: Importación de librerías

**¿Qué hace esta celda?** Carga todas las herramientas que usaremos. Es como
sacar las herramientas del maletín antes de trabajar.

In [ ]:
# ============================================
# CELDA 1: Importación de librerías
# ============================================

# --- Herramientas clásicas (ya las conoces de módulos anteriores) ---
import numpy as np
#   numpy = cálculo numérico. El alias 'np' es por convención.
import pandas as pd
#   pandas = manejo de tablas (DataFrames). El alias 'pd' es por convención.
import matplotlib.pyplot as plt
#   matplotlib = gráficos. 'pyplot' es su módulo de funciones tipo plt.plot().
import seaborn as sns
#   seaborn = gráficos estadísticos más bonitos.

# --- Preprocesamiento (ya las conoces) ---
from sklearn.preprocessing import StandardScaler
#   StandardScaler = el escalador Z-Score. OBLIGATORIO en redes neuronales.
from sklearn.model_selection import train_test_split
#   train_test_split = NUEVA. Divide los datos en entrenamiento y validación.

# --- TensorFlow / Keras (NUEVO HOY) ---
import tensorflow as tf
#   tensorflow = la plataforma de deep learning de Google. 'tf' es su alias.
from tensorflow.keras.models import Sequential
#   Sequential = forma de construir redes apilando capas una tras otra, en orden.
from tensorflow.keras.layers import Dense
#   Dense = capa donde CADA neurona se conecta a TODAS las de la capa anterior.
#   Es la capa fundamental de un MLP.

# --- Estética de los gráficos ---
sns.set_theme(style="whitegrid")
#   Aplica un tema de fondo blanco con cuadrícula a todos los gráficos.

print("Librerías importadas. TensorFlow versión:", tf.__version__)


### ¿Qué hace cada librería NUEVA?

| Librería | ¿Qué es? | ¿Para qué la usamos HOY? |
|----------|----------|---------------------------|
| `tensorflow` | Plataforma de deep learning | El motor que entrena la red |
| `Sequential` | Modelo apilado | Construir la red capa por capa |
| `Dense` | Capa totalmente conectada | Cada neurona se conecta a todas las de la capa anterior |
| `train_test_split` | Divisor de datos | 80% para entrenar, 20% para validar |

> **¿Por qué TensorFlow y no sklearn?** sklearn tiene ML clásico (K-Means,
> árboles, regresión). Para redes neuronales, TensorFlow/Keras es el estándar
> de la industria.

---
## PARTE 2: Cargando y entendiendo los datos

Dataset: `registro_petrofisico.csv` — 7,000 registros de un pozo.
Cada registro tiene 4 curvas geofísicas + la profundidad.

Nuestra tarea: usar **3 variables de entrada** para predecir **1 variable objetivo**:
- Entradas (X): `GR_API`, `ILD_ohm_m`, `RHOB_g_cc`
- Objetivo (y): `NPHI_v_v` (porosidad neutrón)

### Celda 2: Cargar y explorar el dataset

**¿Qué hace esta celda?** Descarga el archivo, lo carga en una tabla, y muestra
información básica para verificar que todo esté en orden.

In [ ]:
# ============================================
# CELDA 2: Cargar el dataset petrofísico
# ============================================

# --- Descargar el archivo desde GitHub ---
!wget -q https://raw.githubusercontent.com/DavidPonce84/machine-learning-course/main/modulo_6_deep_learning/data/registro_petrofisico.csv -O registro_petrofisico.csv
#   '!' ejecuta un comando de Linux en Colab. 'wget' descarga archivos de internet.
#   '-O registro_petrofisico.csv' = guarda con ese nombre.

# --- Cargar en un DataFrame ---
df_well = pd.read_csv('registro_petrofisico.csv')
#   Lee el CSV y lo convierte en una tabla (DataFrame) de pandas.

# --- Exploración inicial ---
print("Dimensiones:", df_well.shape)
#   '.shape' = (filas, columnas). Esperamos (7001, 5): 7,000 registros, 5 columnas.

print("\nTipos de datos:")
print(df_well.dtypes)
#   '.dtypes' = tipo de cada columna. Todas deben ser float64 (decimales).

print("\nEstadísticas:")
df_well.describe()
#   '.describe()' = resumen: media, mín, máx, etc. de cada columna.
#   Fíjate en los rangos: GR (10-120), ILD (0.5-60), RHOB (2.1-2.7), NPHI (0-0.35).

df_well.head()
#   Primeras 5 filas para ver la estructura real.


### ¿Qué buscamos?

- `shape` -> (7001, 5). Si es muy distinto, el CSV no cargó bien.
- `dtypes` -> todas float64. Si alguna es 'object', hay texto infiltrado.
- `describe()` -> los rangos. Nota que **NPHI** (0 a 0.35) es mucho más pequeña
  que **GR** (10 a 120). Esto será clave al escalar.

---
## PARTE 3: Escalar entradas Y salida

**Regla de oro:** las redes neuronales son MUY sensibles a la escala.
Si GR va de 10-120 y NPHI de 0-0.35, el optimizador tendrá problemas para
converger. Por eso escalamos TANTO las entradas como la salida.

> **Diferencia con módulos anteriores:** antes solo escalábamos las entradas.
> En redes neuronales también escalamos la SALIDA (y).

**¿Por qué?** La red genera valores internos pequeños. Si la salida real está
en una escala muy distinta, el cálculo del error se desbalancea.

### Celda 3: Escalar y dividir los datos

**¿Qué hace esta celda?** Escala las entradas y la salida (por separado), y
divide todo en 80% entrenamiento y 20% validación. También guarda la
profundidad para graficarla al final.

In [ ]:
# ============================================
# CELDA 3: Escalar entradas y salida, dividir datos
# ============================================

# --- Definir qué columnas son entrada y cuál es objetivo ---
features = ['GR_API', 'ILD_ohm_m', 'RHOB_g_cc']
#   Las 3 variables de ENTRADA (lo que la red "ve").
target = ['NPHI_v_v']
#   La 1 variable OBJETIVO (lo que la red debe predecir).

# --- Escalar entradas (X) ---
scaler_X = StandardScaler()
#   Crea el escalador para las entradas (aún no hace nada).
X_scaled = scaler_X.fit_transform(df_well[features])
#   '.fit_transform()' = calcula media/desv. y aplica z = (x-media)/desv.

# --- Escalar salida (y) ---
scaler_y = StandardScaler()
#   Crea un escalador SEPARADO para la salida (no debe compartir el de X).
y_scaled = scaler_y.fit_transform(df_well[target])
#   Escala NPHI. Guardamos scaler_y para des-escalar las predicciones al final.

# --- Guardar la profundidad (la usaremos para graficar al final) ---
profundidad = df_well['Profundidad_m'].values
#   Extraemos la columna de profundidad como un array simple.

# --- Dividir TODO en entrenamiento (80%) y validación (20%) ---
X_train, X_val, y_train, y_val, prof_train, prof_val = train_test_split(
    X_scaled, y_scaled, profundidad,
    test_size=0.2,       # 20% para validación
    random_state=42      # semilla fija para que la división sea reproducible
)
#   'train_test_split' baraja y divide. Le pasamos X, y Y profundidad juntos
#   para que los tres queden alineados con la MISMA división.
#   Devuelve 6 arrays: X_train, X_val, y_train, y_val, prof_train, prof_val.

print(f"Entrenamiento: {X_train.shape[0]} registros")
print(f"Validación: {X_val.shape[0]} registros")
#   80% de 7000 = 5600 para entrenar, 1400 para validar.


### ¿Por qué escalar la salida también?

La red usa gradientes (pendientes) para ajustar pesos. Si la salida real es
0.20 y la red predice 0.15, el error es pequeño. Pero si comparamos NPHI (0-0.35)
con GR (10-120) sin escalar, las escalas chocan y el optimizador no converge.
Escalar TODO a media=0 y desv.=1 hace que la red trabaje en un espacio uniforme.

---
## PARTE 4: Construyendo el MLP

Un MLP (Perceptrón Multicapa) es una pila de capas `Dense`.
Cada capa `Dense` conecta todas las neuronas de la capa anterior con las de la siguiente.

Arquitectura de hoy:
1. **Entrada** (implícita): 3 valores (GR, ILD, RHOB)
2. **Capa oculta 1**: 64 neuronas, activación ReLU
3. **Capa oculta 2**: 32 neuronas, activación ReLU
4. **Salida**: 1 neurona, activación lineal (para regresión)

**¿Por qué 64 y 32?** Es un "embudo": se va reduciendo de 3 entradas a 1 salida.
El número exacto es una decisión de diseño que se ajusta por prueba y error.

### Celda 4: Construir el modelo

**¿Qué hace esta celda?** Crea la red apilando 3 capas. La primera recibe 3
entradas, la última produce 1 salida.

In [ ]:
# ============================================
# CELDA 4: Construir el MLP con Keras Sequential
# ============================================

model = Sequential([
    # --- Capa oculta 1: 64 neuronas ---
    Dense(64, activation='relu', input_shape=(3,)),
    #   'Dense(64)' = 64 neuronas en esta capa.
    #   'activation=relu' = ReLU: si la suma es negativa da 0, si es positiva la pasa.
    #   'input_shape=(3,)' = SOLO en la primera capa: le dice a Keras que la
    #   entrada tiene 3 columnas (GR, ILD, RHOB).

    # --- Capa oculta 2: 32 neuronas ---
    Dense(32, activation='relu'),
    #   Segunda capa oculta. No necesita input_shape: Keras lo deduce de la capa anterior.

    # --- Capa de salida: 1 neurona ---
    Dense(1, activation='linear')
    #   'Dense(1)' = 1 neurona de salida (predice UN solo valor: NPHI).
    #   'activation=linear' = pasa la suma tal cual. Para REGRESIÓN usamos lineal.
    #   Para CLASIFICACIÓN usaríamos 'sigmoid' o 'softmax'.
])

model.summary()
#   Muestra la arquitectura: capas, neuronas y cuántos parámetros entrenables hay.


### ¿Por qué ReLU en capas ocultas y Lineal en la salida?

- **ReLU**: si z > 0 -> z; si z < 0 -> 0. Añade no-linealidad, permitiendo que
  la red aprenda patrones curvos. Es la activación estándar en capas ocultas.
- **Lineal**: devuelve z sin cambios. Para regresión queremos predecir cualquier
  valor continuo (porosidad), sin restringirlo a un rango [0,1].

**¿Qué son los parámetros del summary?** Son los pesos y sesgos que la red
aprenderá. En total ~2,369. Cada uno se ajusta durante el entrenamiento.

### Celda 5: Compilar el modelo

**¿Qué hace esta celda?** Configura DOS cosas: el optimizador (cómo ajusta
pesos) y la pérdida (cómo mide el error). Sin compilar, la red no puede entrenar.

In [ ]:
# ============================================
# CELDA 5: Compilar — definir optimizador y pérdida
# ============================================

model.compile(
    optimizer='adam',   # cómo la red ajusta sus pesos en cada paso
    loss='mse'          # error cuadrático medio: qué tan lejos está la predicción
)
#   'optimizer=adam' = el optimizador Adam (veremos más en el Día 2).
#   'loss=mse' = Mean Squared Error. Para regresión es la pérdida estándar.

print("Modelo compilado.")


### ¿Qué significan adam y mse?

- **adam** = el algoritmo que decide cuánto ajustar cada peso. Tiene tasa de
  aprendizaje adaptativa (es el estándar).
- **mse** = Mean Squared Error (error cuadrático medio). Mide qué tan lejos está
  la predicción del valor real, penalizando más los errores grandes.

---
## PARTE 6: Entrenando la red

Entrenar = repetir el ciclo **forward (predecir) -> error -> backpropagation (corregir)**
muchas veces. Cada pasada completa por todos los datos se llama **época**.

**En lenguaje simple:** es como lanzar un dardo (predecir), ver qué tan lejos
quedaste (error), y ajustar tu brazo (corregir). Repites hasta acertar.

### Celda 6: Entrenar el modelo

**¿Qué hace esta celda?** Ejecuta el ciclo de aprendizaje 50 veces (50 épocas).
En cada época, la red ve TODOS los datos, mide su error, y ajusta sus pesos.

In [ ]:
# ============================================
# CELDA 6: Entrenar la red por 50 épocas
# ============================================

history = model.fit(
    X_train, y_train,                       # datos de entrenamiento
    epochs=50,                              # 50 pasadas completas por los datos
    batch_size=32,                          # procesa 32 registros antes de actualizar pesos
    validation_data=(X_val, y_val),         # evalúa en validación cada época
    verbose=1                               # muestra el progreso en pantalla
)
#   'model.fit' = entrena la red.
#   'epochs=50' = 50 vueltas completas por los datos de entrenamiento.
#   'batch_size=32' = actualiza pesos cada 32 registros (mini-batch).
#   'validation_data' = datos que la red NO usa para entrenar, solo para medir
#   si generaliza bien. Clave para detectar overfitting.
#   'history' = objeto que guarda la pérdida de cada época para graficarla.


### ¿Qué verás mientras entrena?

Keras imprimirá una línea por época con la pérdida (loss) y la pérdida de
validación (val_loss). Deberías ver cómo ambos números BAJAN con las épocas.
Eso significa que la red está aprendiendo.

### Celda 7: Graficar la curva de pérdida

**¿Qué hace esta celda?** Dibuja cómo bajó el error en cada época, para el
entrenamiento y para la validación. Es la forma visual de confirmar que aprendió.

In [ ]:
# ============================================
# CELDA 7: Curva de pérdida (MSE) por época
# ============================================

plt.figure(figsize=(10, 5))
#   Crea un lienzo de 10x5 pulgadas.

plt.plot(history.history['loss'], label='Entrenamiento', linewidth=2)
#   'history.history' es un diccionario con las métricas registradas.
#   '['loss']' = la pérdida de ENTRENAMIENTO en cada época.
plt.plot(history.history['val_loss'], label='Validación', linewidth=2)
#   '['val_loss']' = la pérdida de VALIDACIÓN en cada época.

plt.title('Evolución de la Pérdida (MSE) durante el Entrenamiento')
plt.xlabel('Época')
plt.ylabel('MSE (Error Cuadrático Medio)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
#   Si AMBAS curvas bajan juntas -> la red aprende bien.
#   Si entrenamiento baja pero validación sube -> OVERFITTING (Día 2).


### ¿Qué esperamos ver?

Ambas curvas deberían **bajar juntas** y estabilizarse. Eso significa que la
red aprendió la relación GR/ILD/RHOB -> NPHI y generaliza bien.

> Si ves que la de validación empieza a SUBIR mientras la de entrenamiento sigue
> bajando, eso es **overfitting**. Lo atacaremos en el Día 2.

---
## PARTE 8: Predecir y des-escalar

La red predice en espacio escalado (media=0, desv.=1). Para comparar con la
porosidad REAL, debemos **des-escalar** (invertir la transformación Z-Score).

**En lenguaje simple:** la red habla en "idioma escalado". Para que un
petrofísico entienda, traducimos de vuelta a porosidad real (0 a 0.35).

### Celda 8: Predecir y des-escalar

**¿Qué hace esta celda?** Pide a la red que prediga sobre los datos de
validación, y traduce el resultado a unidades reales.

In [ ]:
# ============================================
# CELDA 8: Predecir porosidad y des-escalar
# ============================================

# --- Predecir sobre los datos de validación ---
y_pred_scaled = model.predict(X_val)
#   'model.predict' = la red genera predicciones para los datos de validación.
#   'y_pred_scaled' = predicciones en espacio ESCALADO (media=0, desv.=1).

# --- Des-escalar las predicciones ---
y_pred = scaler_y.inverse_transform(y_pred_scaled)
#   'inverse_transform' = revierte el Z-Score: x = z * desv + media.
#   Volvemos a las unidades reales de porosidad (fracción, ej. 0.20 = 20%).

# --- Des-escalar los valores reales para comparar ---
y_val_real = scaler_y.inverse_transform(y_val)
#   También des-escalamos los valores reales de validación.

print(f"Predicción completada: {y_pred.shape[0]} valores.")
print(f"Rango de porosidad predicha: {y_pred.min():.3f} - {y_pred.max():.3f}")
print(f"Rango de porosidad real: {y_val_real.min():.3f} - {y_val_real.max():.3f}")
#   Comparamos rangos: si son similares, la red predice en la escala correcta.


### ¿Por qué des-escalar?

La red aprende en espacio escalado porque así converge mejor. Pero un valor
escalado de '0.5' no significa nada para un petrofísico. Al des-escalar,
volvemos a porosidad real (fracción de 0 a 0.35).

### Celda 9: Comparar predicción vs realidad

**¿Qué hace esta celda?** Grafica la porosidad real (azul) y la predicha (naranja)
a lo largo de la profundidad. Si se solapan, la red aprendió la física.

In [ ]:
# ============================================
# CELDA 9: Visualizar porosidad predicha vs real
# ============================================

plt.figure(figsize=(10, 6))

plt.scatter(prof_val, y_val_real, s=3, alpha=0.5, label='NPHI Real', color='#38bdf8')
#   'prof_val' = la profundidad de validación (ya guardada en la Celda 3).
#   'y_val_real' = la porosidad REAL. Puntos azules.
plt.scatter(prof_val, y_pred, s=3, alpha=0.5, label='NPHI Predicha', color='#f59e0b')
#   'y_pred' = la porosidad PREDICHA. Puntos naranjas.

plt.xlabel('Profundidad (m)')
plt.ylabel('Porosidad NPHI (v/v)')
plt.title('Porosidad Real vs Predicha por la Red Neuronal')
plt.legend()
plt.show()

#   Si azul y naranja se SOLAPAN -> la red aprendió la física.
#   Si están dispersos -> la red no aprendió bien.


---
## RECAP: El Pipeline MLP Completo

```
1. IMPORTAR      -> tensorflow, Keras, sklearn
2. CARGAR datos  -> pd.read_csv()
3. ESCALAR X e y -> StandardScaler() (ambos)
4. DIVIDIR       -> train_test_split() 80/20
5. CONSTRUIR     -> Sequential([Dense, Dense, Dense])
6. COMPILAR      -> optimizer='adam', loss='mse'
7. ENTRENAR      -> model.fit(epochs=50)
8. GRAFICAR      -> curva de pérdida
9. PREDECIR      -> predict() + inverse_transform()
```

### Lo que aprendiste hoy

- Una neurona = pesos + sesgo + activación
- Un MLP = capas Dense apiladas
- ReLU en capas ocultas, Lineal en salida de regresión
- Entrenar = repetir forward -> error -> backpropagation
- Escalar SIEMPRE (entradas Y salida)

> **Siguiente:** Mañana veremos el enemigo #1 del deep learning: el overfitting,
> y cómo combatirlo con Dropout y Early Stopping.

---
## PARTE 9: Laboratorio Interactivo — Explora las Neuronas

Ahora te toca a ti. Usa el slider para cambiar el número de neuronas y observa
cómo cambia la curva de pérdida. Esta es la forma de "sentir" el efecto de la
capacidad del modelo.

**En lenguaje simple:** mueve el slider y mira.
- Pocas neuronas: la red no aprende bien (error alto en ambas curvas).
- Muchas neuronas: empieza a memorizar (validación sube).
- El equilibrio está en el medio.


In [ ]:
# ============================================
# CELDA 10 (INTERACTIVA): Laboratorio de neuronas
# ============================================

import ipywidgets as widgets
from IPython.display import clear_output
#   ipywidgets = sliders y controles interactivos.

def laboratorio(neuronas=64, epochs=40):
    clear_output(wait=True)  # limpia la salida anterior para no acumular gráficas
    model = Sequential([
        Dense(neuronas, activation='relu', input_shape=(3,)),
        Dense(neuronas // 2, activation='relu'),   # la segunda capa usa la mitad
        Dense(1, activation='linear')
    ])
    model.compile(optimizer='adam', loss='mse')
    history = model.fit(X_train, y_train, epochs=epochs,
                        validation_data=(X_val, y_val), verbose=0)
    plt.figure(figsize=(10, 4))
    plt.plot(history.history['loss'], label='Entrenamiento')
    plt.plot(history.history['val_loss'], label='Validación')
    plt.title(f'Curva de pérdida con {neuronas} neuronas')
    plt.xlabel('Época'); plt.ylabel('MSE')
    plt.legend(); plt.show()
    print(f'Pérdida final de validación: {history.history["val_loss"][-1]:.4f}')

widgets.interact(laboratorio,
                 neuronas=widgets.IntSlider(8, 128, step=8, value=64),
                 epochs=widgets.IntSlider(10, 100, step=10, value=40))
#   Mueve 'neuronas' y mira cómo cambia la curva.
#   Con pocas (8-16): error alto = no aprende.
#   Con muchas (120-128): validación sube = memoriza.
